### PyTorch
- [https://pytorch.org/](https://pytorch.org/)

### diffusers
- [https://github.com/huggingface/diffusers](https://github.com/huggingface/diffusers)

### Stability AI
- [https://huggingface.co/stabilityai](https://huggingface.co/stabilityai)

### 用來申請授權的範例: stable-diffusion-xl-base-1.0
- [https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0](https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0)

### Negative Prompts 全攻略
- [https://blog.256pages.com/negative-prompts/](https://blog.256pages.com/negative-prompts/)

### 圖像生成是如何運作？用Pytorch實作Stable Diffusion！
- [https://edge.aif.tw/express-stable-diffusion/](https://edge.aif.tw/express-stable-diffusion/)

### SDXL 模型之 base、refiner 和 VAE
- [https://blog.csdn.net/mogoweb/article/details/132114260](https://blog.csdn.net/mogoweb/article/details/132114260)

### SDXL + Refiner
- [https://medium.com/@wangdk93/sdxl-refiner-9b9694d0b7a8](https://medium.com/@wangdk93/sdxl-refiner-9b9694d0b7a8)

### 古風模型
- [https://huggingface.co/John6666](https://huggingface.co/John6666)
- [https://huggingface.co/John6666/iniverse-mix-xl-sfwnsfw-pony-guofeng-v30-sdxl](https://huggingface.co/John6666/iniverse-mix-xl-sfwnsfw-pony-guofeng-v30-sdxl)

### 模型作者的站台
- [https://civitai.com/models/226533/iniverse-mixsfw-and-nsfw](https://civitai.com/models/226533/iniverse-mixsfw-and-nsfw)

### Civitai
- [連結](https://civitai.com/)
  - 選擇 Featured Models，最右有 Explore all models 的連結。
  - 在預覽圖上，右上角有類似一支筆 (Create 按鈕)，就可以點進去看。
  - 進入內頁後，可以看到預覽器的右上角有一個 Create 按鈕，點下去就會在左側出現設定的地方。
  - 生成圖片前，參考以下範例的 Prompt 與 Negative Prompt，將它們貼到左側的設定區域。
  - 也可以在網頁最上方的搜尋欄，選擇 Models，輸入 CN，可以找到東方臉孔的模型輸出結果。

#### Stable Diffusion 提示詞 (prompt) 參考資料
- [PormptHero](https://prompthero.com/)
  - 選上方連結 [Stable Diffusion](https://prompthero.com/stable-diffusion-prompts)，尋找偏好的圖片，點進去參考它們的設定
  - [Prompt 與 Negative prompt](https://prompthero.com/prompt/7fc0b9928fb-stable-diffusion-xl-base-0-9-playtime-s-over)

#### 範例
Prompt:
```log
(masterpiece, best quality:1.2<),highly detailed,extremely detailed,real photo,
fullbody,1girl,solo,asian,looking at viewer,(body facing viewer:1.2)(relax sitting),knees separation,
red lips,brown long hair,
collared shirt and dress shirt,long sleeves,(knees length dress:1.1),
(wrap hip very thick pantyhose:1.1),color high heels,
nice figure,good anatomy,good proportions,nice pose,(2shoes,2legs:1.2)(perfect legs:1.1),nice hand,
outdoors,buildings,photorealistic,realistic,<lora:more_details:0.4>,
<lora:yuzuv10:0.5>,<lora:sit_cross_leg_v2:0.6>,<lora:control_skin_exposure:-1.0>
```

```log
a chinese young lady lying of a pink wall with a bow tie on it's chest and a pink and white top, white shorts, sunny, a screenshot, aestheticism, movie style, studio lighting, movie cinematic,  horizon,
1girl, solo, ,masterpiece, high quality, highres, absurdres, high details,8k,HDR,raw photo,realistic, bokeh, shallow depth of field, beautiful eyes, high detail eyes, beautiful face, high detail face, high detail skin, beautiful hands, beautiful fingers, beautiful eyelashes, fingernails,(above the thigh:1.6), pixie curly cut hair, lying down, hand reaching towards viewer,  big pink towel, white simple background
```

Negative Prompt:
```log
bad anatomy, lowres, normal quality, grayscale, worstquality, watermark, bad proportions, out of focus, long neck, deformed, mutated, mutation, disfigured, poorly drawn face, skin blemishes, skin spots, acnes, missing limb, malformed limbs, floating limbs, disconnected limbs, extra limb, extra arms, mutated hands, poorly drawn hands, malformed hands, mutated hands and fingers, bad hands, missing fingers, fused fingers, too many fingers, extra legs, bad feet, cross-eyed, (distorted, :1.3) , (:1.4) , low quality, camera, BadDream, UnrealisticDream, bad-hands-5, BadNegAnatomyV1-neg, EasyNegative, FastNegativeV2, bad-picture-chill-75v
```

In [ ]:
# 安裝 diffusers
!pip install --upgrade diffusers[torch]

# 安裝 huggingface 官方下載工具
!pip install huggingface_hub

In [ ]:
# 匯入 huggingface_hub 的 snapshot_download 函式
from huggingface_hub import snapshot_download

# 設定你要下載的模型代號與本地存放路徑
model_id = "John6666/iniverse-mix-xl-sfwnsfw-pony-guofeng-v30-sdxl"
local_model_path = "/content/iniverse_model"

print(f"正在下載模型至 {local_model_path}，請稍候...")

# 開始下載（這會把整個 Repo 結構完整下載下來）
snapshot_download(
    repo_id=model_id,
    local_dir=local_model_path,
    local_dir_use_symlinks=False, # 確保下載的是實體檔案，而非軟連結
    # token="你的Huggingface_Token" # 如果該模型需要 Token 驗證，請把這行註解解開並填入 Token
)

print("模型下載完成！")

In [ ]:
# 匯入 diffusers 的相關模組
from diffusers import DiffusionPipeline, DPMSolverMultistepScheduler

# 匯入 torch，確保在 T4 GPU 上能夠順利運行，
# 建議使用 torch.float16 的數據類型，這樣可以大幅減少模型的記憶體使用量
import torch

# 為了避免在 T4 GPU 上遇到 OOM 問題，
# 建議開啟 PyTorch 的 Dynamo 錯誤抑制功能，讓模型能夠順利運行
import torch._dynamo
torch._dynamo.config.suppress_errors = True

# 關鍵修改：將原本的模型代號，改成剛剛下載的本地端路徑
local_model_path = "/content/iniverse_model"

# 設定 Scheduler (改從本地路徑讀取)
ddim = DPMSolverMultistepScheduler.from_pretrained(
    local_model_path, 
    subfolder="scheduler"
)

# 讀取預訓練模型 (改從本地路徑讀取，不需要再連網下載或驗證 Token)
pipe = DiffusionPipeline.from_pretrained(
    local_model_path,
    torch_dtype=torch.float16,
    use_safetensors=True,
    scheduler=ddim,
    # variant="fp16"# 如果模型有提供 fp16 版本，可以加上這行參數，讓模型直接以 fp16 的格式載入，進一步減少記憶體使用量
)

# 將讀取的模型放到 GPU memory 當中
pipe = pipe.to("cuda")

# 針對 T4 GPU 的記憶體優化（GPU 的 Memory 不夠時，可以解開這行註解，避免 SDXL 模型在 T4 上 OOM）
# pipe.enable_model_cpu_offload()

### num_inference_steps
- 一般來說，使用的 step 越多，結果越好，但是 step 越多，所需的生成時間就越長。
- 由於 Stable Diffusion 在 step 相對少的情況下效果很好，通常建議使用預設的數值為 50。
- 想要更快的結果，可以使用較小的 step；想要更高品質的結果，可以使用更大的數字。

### guidance_scale
- Guidance Scale 是生成圖片與輸入提示的緊密程度，及輸入的多樣性間的平衡，它的典型值在 7.5 左右。
- 增加的比例越多，圖片的質量就越高，但是得到的多樣性就越低。

In [ ]:
# 提示詞設定
prompt = '''(hanfu:1.4),(score_9,score_8_up,score_7_up),18-year-old chinese hanfu girl,realistic,oval face,backlight,chinese hair pin,(a faint smile:0.8),'''
negative_prompt = '''(score_6,score_5,score_4),ugly,disfigured,poorly drawn face,malformed hands,mutated hands,poorly drawn hands,mutated hands and fingers,bad hands,missing fingers,fused fingers,too many fingers,'''

# 使用 pipeline 生成
print("正在生成圖片...")
'''

'''
image = pipe(
    prompt,
    negative_prompt=negative_prompt,
    num_inference_steps=50,
    width=1024,
    height=1024,
    guidance_scale=5,
    generator=torch.Generator("cuda").manual_seed(42) # 設定種子，讓每次生成的圖片都一樣（方便測試和調整參數）
).images[0]

# 預覽結果
image

In [ ]:
import torch
import gc

# 刪除與模型相關的變數名稱
if 'pipe' in locals():
    del pipe
if 'image' in locals():
    del image

# 強制執行 Python 的垃圾回收，清空系統記憶體中的殘留物件
gc.collect()

# 強制清除 PyTorch 在 GPU 內暫存的 VRAM Buffer
torch.cuda.empty_cache()

# 檢查目前剩餘的顯存狀況
print(f"目前已分配的 VRAM: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"當前已保留的 VRAM: {torch.cuda.memory_reserved() / 1024**3:.2f} GB")
print("GPU 的 VRAM 已經清理完成！")